In [ ]:
import pandas as pd 
import numpy 
import matplotlib.pyplot as plt 

In [ ]:
df =pd.read_csv("spotify_top_songs_audio_features.csv")
df.head()

In [ ]:
median_weeks = df['weeks_on_chart'].median()
df['hit'] = (df['weeks_on_chart']>=median_weeks).astype(int)



In [ ]:
numeric_cols = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence",
    "loudness", "tempo", "duration_ms"
]

categorical_cols = ["key", "mode", "time_signature"]


In [ ]:
X = df[numeric_cols + ['key', 'mode', 'time_signature']]
y = df['hit']

In [ ]:
from sklearn.model_selection import train_test_split ,GridSearchCV,StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler ,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

In [ ]:
X_train,X_test,y_train,y_test =train_test_split(X,y,test_size=0.2,random_state=11,stratify=y)

preprocessor = ColumnTransformer([
    ('num',StandardScaler(),numeric_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_cols)
])

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__penalty": ["l1", "l2"],
    "classifier__solver": ["liblinear"]
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

gs_lr =GridSearchCV(
    lr_pipeline,
    lr_param_grid,
    cv=cv,
    scoring = 'accuracy',
    n_jobs=-1
)


gs_lr.fit(X_train,y_train)

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('classifier', RandomForestClassifier(random_state=11))
])


rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5]
}
gs_rf = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv= cv,
    scoring = 'accuracy',
    n_jobs = -1
)

gs_rf.fit(X_train,y_train)



training accuracy

In [ ]:
lr_train_acc = accuracy_score(
    y_train , gs_lr.predict(X_train)
)
rf_train_acc = accuracy_score(
    y_train,
    gs_rf.predict(X_train)
)

test accuracy 

In [ ]:
lr_test_acc = accuracy_score(
    y_test,
    gs_lr.predict(X_test)
)

rf_test_acc = accuracy_score(
    y_test,
    gs_rf.predict(X_test)
)

In [ ]:

print("Best LR Params :", gs_lr.best_params_)
print("Best RF Params :", gs_rf.best_params_)


In [ ]:
print("\nLogistic Regression")
print("Train Accuracy :", round(lr_train_acc,4))
print("Test Accuracy  :", round(lr_test_acc,4))




In [ ]:
print("\nRandom Forest")
print("Train Accuracy :", round(rf_train_acc,4))
print("Test Accuracy  :", round(rf_test_acc,4))

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42
    ))
])

xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 4, 5],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

In [ ]:
gs_xgb = GridSearchCV(
    xgb_pipeline,
    xgb_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

gs_xgb.fit(X_train, y_train)

In [ ]:
xgb_train_acc = accuracy_score(
    y_train,
    gs_xgb.predict(X_train)
)

xgb_test_acc = accuracy_score(
    y_test,
    gs_xgb.predict(X_test)
)

print("Best XGBoost Params:", gs_xgb.best_params_)
print("Training Accuracy :", round(xgb_train_acc, 4))
print("Testing Accuracy  :", round(xgb_test_acc, 4))

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    ["Logistic Regression", "Random Forest", "XGBoost"],
    [lr_train_acc, rf_train_acc, xgb_train_acc],
    marker='o',
    label='Training Accuracy'
)

plt.plot(
    ["Logistic Regression", "Random Forest", "XGBoost"],
    [lr_test_acc, rf_test_acc, xgb_test_acc],
    marker='o',
    label='Testing Accuracy'
)

plt.title("Training vs Testing Accuracy")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pickle 

with open ("best_model.pkl","wb") as f:
    pickle.dump(gs_rf.best_estimator_,f)


print("best model saved")


In [ ]:
best_rf = gs_rf.best_estimator_

rf = best_rf.named_steps['classifier']

importances = rf.feature_importances_

feature_names = (
    numeric_cols +
    list(
        best_rf.named_steps['preprocessor']
        .named_transformers_['cat']
        .get_feature_names_out(categorical_cols)
    )
)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(importance_df.head(15))